# TASK 01 Nested Transaction Analyzer

In [ ]:
from collections import defaultdict
from numbers import Real


def analyze_transactions(transactions):
    
    if transactions is None or isinstance(transactions, (str, bytes, dict)):
        raise TypeError("transactions must be a list of dictionaries")

    required_keys = {"id", "customer", "category", "amount", "status"}
    totals = {}
    category_spend = defaultdict(lambda: defaultdict(int))
    seen_ids = set()

    for position, transaction in enumerate(transactions):
        if not isinstance(transaction, dict):
            raise ValueError(f"Transaction at position {position} must be a dictionary")
        missing = required_keys - transaction.keys()
        if missing:
            raise ValueError(f"Transaction at position {position} is missing keys: {sorted(missing)}")

        transaction_id = transaction["id"]
        customer = transaction["customer"]
        category = transaction["category"]
        amount = transaction["amount"]
        status = str(transaction["status"]).strip().lower()

        if isinstance(amount, bool) or not isinstance(amount, Real) or amount != amount or amount < 0:
            raise ValueError(f"Transaction {transaction_id!r}: amount must be a non-negative number")

        if transaction_id in seen_ids:
            continue                      
        seen_ids.add(transaction_id)

        totals.setdefault(customer, 0)   
        if status == "success":
            totals[customer] += amount
            category_spend[customer][category] += amount

    ordered = sorted(totals.items(), key=lambda item: (-item[1], item[0]))

    report = []
    previous_total, rank = None, 0
    for position, (customer, total) in enumerate(ordered, start=1):
        if total != previous_total:
            rank, previous_total = position, total
        spend = category_spend.get(customer)
        top_category = None
        if spend:
            
            top_category = min(spend.items(), key=lambda item: (-item[1], str(item[0])))[0]
        report.append({"customer": customer, "total": total, "top_category": top_category, "rank": rank})
    return report



sample = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
]
report = analyze_transactions(sample)
for row in report:
    print(row)
assert [(r["customer"], r["total"]) for r in report] == [("Sara", 700), ("Ali", 500)]

edge = [
    {"id": 1, "customer": "Zed", "category": "Food", "amount": 100, "status": "success"},
    {"id": 1, "customer": "Zed", "category": "Food", "amount": 100, "status": "success"},   
    {"id": 2, "customer": "Amy", "category": "Food", "amount": 100, "status": "success"},
    {"id": 3, "customer": "Bob", "category": "Travel", "amount": 50, "status": "failed"},   
    {"id": 4, "customer": "Amy", "category": "Travel", "amount": 300, "status": "success"},
    {"id": 5, "customer": "Cy", "category": "Bills", "amount": 100, "status": "success"},   
]
print()
edge_report = analyze_transactions(edge)
for row in edge_report:
    print(row)
assert [(r["customer"], r["total"], r["rank"]) for r in edge_report] == [
    ("Amy", 400, 1), ("Cy", 100, 2), ("Zed", 100, 2), ("Bob", 0, 4)]
assert edge_report[0]["top_category"] == "Travel" and edge_report[-1]["top_category"] is None
assert analyze_transactions([]) == []
print("\nTask 01: all checks passed")


{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}

{'customer': 'Amy', 'total': 400, 'top_category': 'Travel', 'rank': 1}
{'customer': 'Cy', 'total': 100, 'top_category': 'Bills', 'rank': 2}
{'customer': 'Zed', 'total': 100, 'top_category': 'Food', 'rank': 2}
{'customer': 'Bob', 'total': 0, 'top_category': None, 'rank': 4}

Task 01: all checks passed


# TASK 02 Longest Valid Window

In [ ]:
def longest_unique_window(text):
    
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    last_seen = {}
    left = best_start = best_length = 0

    for right, char in enumerate(text):
        if char in last_seen and last_seen[char] >= left:
            left = last_seen[char] + 1
        last_seen[char] = right

        window_length = right - left + 1
        if window_length > best_length:
            best_start, best_length = left, window_length

    return text[best_start:best_start + best_length], best_start, best_length



import random

assert longest_unique_window("abcabcbb") == ("abc", 0, 3)
assert longest_unique_window("") == ("", 0, 0)
assert longest_unique_window("bbbbb") == ("b", 0, 1)
assert longest_unique_window("pwwkew") == ("wke", 2, 3)
assert longest_unique_window("a b a") == ("a b", 0, 3)          
assert longest_unique_window("ab!ab!") == ("ab!", 0, 3)          
assert longest_unique_window("abba") == ("ab", 0, 2)             


def brute_force(text):
    best = ("", 0, 0)
    for i in range(len(text)):
        for j in range(i + 1, len(text) + 1):
            if len(set(text[i:j])) == j - i and j - i > best[2]:
                best = (text[i:j], i, j - i)
    return best


random.seed(7)
for _ in range(300):
    s = "".join(random.choice("abc d!") for _ in range(random.randint(0, 14)))
    assert longest_unique_window(s) == brute_force(s), s

print(longest_unique_window("abcabcbb"), longest_unique_window(""))
print("Task 02: all checks passed (including 300 random comparisons with brute force)")


('abc', 0, 3) ('', 0, 0)
Task 02: all checks passed (including 300 random comparisons with brute force)


# TASK 03 Recursive JSON Path Explorer

In [ ]:
def _strict_equal(left, right):
   
    if isinstance(left, bool) or isinstance(right, bool):
        return isinstance(left, bool) and isinstance(right, bool) and left == right
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(_strict_equal(left[k], right[k]) for k in left)
    if isinstance(left, list) and isinstance(right, list):
        return len(left) == len(right) and all(_strict_equal(a, b) for a, b in zip(left, right))
    return left == right


def find_paths(data, target):
    
    paths = []
    active_containers = set()     

    def visit(value, path):
        if _strict_equal(value, target):
            paths.append(path)
        walk(value, path)

    def walk(node, path):
        if not isinstance(node, (dict, list)):
            return
        if id(node) in active_containers:
            raise ValueError(f"Circular reference detected at '{path}'")
        active_containers.add(id(node))
        try:
            if isinstance(node, dict):
                for key, value in node.items():
                    visit(value, f"{path}.{key}" if path else str(key))
            else:
                for index, value in enumerate(node):
                    visit(value, f"{path}[{index}]")
        finally:
            active_containers.discard(id(node))

    walk(data, "")
    return paths


data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
print(find_paths(data, 5))
assert find_paths(data, 5) == ["a[0].x", "a[1]", "b.y"]


mixed = {"flag": True, "one": 1, "zero": 0, "off": False, "nested": [True, 1]}
assert find_paths(mixed, 1) == ["one", "nested[1]"]
assert find_paths(mixed, True) == ["flag", "nested[0]"]
assert find_paths(mixed, 0) == ["zero"]
assert find_paths(mixed, False) == ["off"]

assert find_paths({}, 5) == [] and find_paths([], 5) == []
assert find_paths([5, [5, {"k": 5}]], 5) == ["[0]", "[1][0]", "[1][1].k"]       
assert find_paths({"a": {"b": {"c": [1, 2, {"d": "hit"}]}}}, "hit") == ["a.b.c[2].d"]
assert find_paths({"a": None, "b": [None]}, None) == ["a", "b[0]"]
assert find_paths({"a": [1, 2], "b": [1, 2]}, [1, 2]) == ["a", "b"]            

loop = {"name": "loop"}
loop["self"] = loop
try:
    find_paths(loop, 1)
except ValueError as error:
    print("Circular reference handled ->", error)

print("Task 03: all checks passed")


['a[0].x', 'a[1]', 'b.y']
Circular reference handled -> Circular reference detected at 'self'
Task 03: all checks passed


# TASK 04 Dependency Execution Planner

In [ ]:
import heapq


def plan_execution(tasks, dependencies):
   
    task_list = list(tasks)
    if not all(isinstance(name, str) for name in task_list):
        raise ValueError("All task names must be strings")
    if len(set(task_list)) != len(task_list):
        raise ValueError("Duplicate task names are not allowed")

    known = set(task_list)
    successors = {name: set() for name in known}
    indegree = {name: 0 for name in known}

    for pair in dependencies:
        try:
            before, after = pair
        except (TypeError, ValueError):
            raise ValueError(f"Invalid dependency {pair!r}: expected a (before, after) pair") from None

        unknown = [name for name in (before, after) if name not in known]
        if unknown:
            raise ValueError(f"Unknown task name(s) {unknown} in dependency {pair!r}")
        if before == after:
            raise ValueError(f"Cycle detected: task '{before}' cannot depend on itself")

        if after not in successors[before]:        
            successors[before].add(after)
            indegree[after] += 1

    ready = [name for name in known if indegree[name] == 0]
    heapq.heapify(ready)
    order = []

    while ready:
        current = heapq.heappop(ready)
        order.append(current)
        for follower in successors[current]:
            indegree[follower] -= 1
            if indegree[follower] == 0:
                heapq.heappush(ready, follower)

    if len(order) != len(known):
        stuck = sorted(name for name in known if indegree[name] > 0)
        raise ValueError("Cycle detected; these tasks are in or blocked by a dependency cycle: " + ", ".join(stuck))
    return order



tasks = ["build", "test", "deploy"]
deps = [("build", "test"), ("test", "deploy")]
print(plan_execution(tasks, deps))
assert plan_execution(tasks, deps) == ["build", "test", "deploy"]

assert plan_execution(["c", "a", "b"], []) == ["a", "b", "c"]                       
assert plan_execution(["d", "b", "c", "a"], [("a", "d"), ("a", "c")]) == ["a", "b", "c", "d"]
assert plan_execution([], []) == []
assert plan_execution(["x", "y"], [("x", "y"), ("x", "y")]) == ["x", "y"]           

for bad_tasks, bad_deps, label in [
    (["a", "b"], [("a", "b"), ("b", "a")], "two-task cycle"),
    (["a", "b", "c"], [("a", "b"), ("b", "c"), ("c", "a")], "three-task cycle"),
    (["a"], [("a", "a")], "self dependency"),
    (["a", "b"], [("a", "zzz")], "unknown task"),
    (["a", "a"], [], "duplicate task"),
    (["a", "b"], [("a",)], "malformed pair"),
]:
    try:
        plan_execution(bad_tasks, bad_deps)
        raise AssertionError(f"{label}: expected ValueError")
    except ValueError as error:
        print(f"{label:<18} -> ValueError: {error}")

print("Task 04: all checks passed")


['build', 'test', 'deploy']
two-task cycle     -> ValueError: Cycle detected; these tasks are in or blocked by a dependency cycle: a, b
three-task cycle   -> ValueError: Cycle detected; these tasks are in or blocked by a dependency cycle: a, b, c
self dependency    -> ValueError: Cycle detected: task 'a' cannot depend on itself
unknown task       -> ValueError: Unknown task name(s) ['zzz'] in dependency ('a', 'zzz')
duplicate task     -> ValueError: Duplicate task names are not allowed
malformed pair     -> ValueError: Invalid dependency ('a',): expected a (before, after) pair
Task 04: all checks passed


# TASK 05 Expression Validator and Evaluator

In [ ]:
class ExpressionError(ValueError):
    """Raised when an expression is malformed."""

_BINARY_OPERATORS = {
    "+": (1, lambda a, b: a + b),
    "-": (1, lambda a, b: a - b),
    "*": (2, lambda a, b: a * b),
    "/": (2, None),
}
_DIGITS = "0123456789"


def _tokenize(expression):
    tokens, index = [], 0
    while index < len(expression):
        char = expression[index]
        if char.isspace():
            index += 1
        elif char in _DIGITS:
            end = index
            while end < len(expression) and expression[end] in _DIGITS:
                end += 1
            tokens.append(("num", int(expression[index:end])))
            index = end
        elif char in _BINARY_OPERATORS:
            tokens.append(("op", char))
            index += 1
        elif char in "()":
            tokens.append((char, char))
            index += 1
        else:
            raise ExpressionError(f"Invalid character {char!r} at position {index}")
    return tokens


def _to_postfix(tokens):
    """Shunting-yard with an operator stack. Unary +/- are recognised by position."""
    output, stack = [], []
    expect_operand = True                     

    for kind, value in tokens:
        if kind == "num":
            if not expect_operand:
                raise ExpressionError("Missing operator before a number")
            output.append(("num", value))
            expect_operand = False

        elif kind == "(":
            if not expect_operand:
                raise ExpressionError("Missing operator before '('")
            stack.append(("(", "("))

        elif kind == ")":
            if expect_operand:
                raise ExpressionError("Missing operand before ')' (empty parentheses or dangling operator)")
            while stack and stack[-1][0] != "(":
                output.append(stack.pop())
            if not stack:
                raise ExpressionError("Unbalanced parentheses: unexpected ')'")
            stack.pop()                         
            expect_operand = False

        else:                                   
            if expect_operand:
                if value in "+-":               
                    stack.append(("unary", value))
                    continue
                raise ExpressionError(f"Operator '{value}' is missing its left operand")
            precedence = _BINARY_OPERATORS[value][0]
            while stack and stack[-1][0] in ("op", "unary") and (
                stack[-1][0] == "unary" or _BINARY_OPERATORS[stack[-1][1]][0] >= precedence
            ):
                output.append(stack.pop())
            stack.append(("op", value))
            expect_operand = True

    if expect_operand:
        raise ExpressionError("Expression is empty or ends with an operator")
    while stack:
        top = stack.pop()
        if top[0] == "(":
            raise ExpressionError("Unbalanced parentheses: missing ')'")
        output.append(top)
    return output


def _evaluate_postfix(postfix):
    stack = []
    for kind, value in postfix:
        if kind == "num":
            stack.append(value)
        elif kind == "unary":
            operand = stack.pop()
            stack.append(-operand if value == "-" else operand)
        else:
            right, left = stack.pop(), stack.pop()
            if value == "/":
                if right == 0:
                    raise ZeroDivisionError("Division by zero in expression")
                stack.append(left / right)
            else:
                stack.append(_BINARY_OPERATORS[value][1](left, right))
    return stack[0]


def evaluate_expression(expression):
    """Evaluate an integer expression with + - * /, parentheses, unary signs and whitespace.

    No eval()/exec(). Raises ExpressionError (a ValueError) for malformed input and
    ZeroDivisionError for division by zero. "/" is true division, so 18 / 3 -> 6.0.
    """
    if not isinstance(expression, str):
        raise TypeError("expression must be a string")
    return _evaluate_postfix(_to_postfix(_tokenize(expression)))



result = evaluate_expression("-(3 + 2) * 4 + 18 / 3")
print(result)
assert result == -14.0

valid_cases = {
    "2 + 3 * 4": 14, "(2 + 3) * 4": 20, "7 - 3 - 2": 2, "100 / 10 / 5": 2.0, "10 / 4": 2.5,
    "2 * -3": -6, "- -3": 3, "-(-(4))": 4, "+5 - -2": 7, "  42  ": 42, "((((7))))": 7,
    "-2 * 3 + 10": 4, "8 / 2 * 4": 16.0, "1 - (2 - (3 - 4))": -2,
}
for text, expected in valid_cases.items():
    assert evaluate_expression(text) == expected, (text, evaluate_expression(text), expected)

for bad in ["", "   ", "2 +", "(2 + 3", "2 + 3)", "2 3", "()", "* 3", "3 $ 4", "2 + * 3", "3.5 + 1", "(2)(3)", "abc"]:
    try:
        evaluate_expression(bad)
        raise AssertionError(f"{bad!r}: expected ExpressionError")
    except ExpressionError as error:
        print(f"{bad!r:<10} -> ExpressionError: {error}")

for zero in ["5 / 0", "5 / (2 - 2)", "1 / (3 - 3) + 4"]:
    try:
        evaluate_expression(zero)
        raise AssertionError("expected ZeroDivisionError")
    except ZeroDivisionError as error:
        print(f"{zero!r:<18} -> ZeroDivisionError: {error}")

print("Task 05: all checks passed")


-14.0
''         -> ExpressionError: Expression is empty or ends with an operator
'   '      -> ExpressionError: Expression is empty or ends with an operator
'2 +'      -> ExpressionError: Expression is empty or ends with an operator
'(2 + 3'   -> ExpressionError: Unbalanced parentheses: missing ')'
'2 + 3)'   -> ExpressionError: Unbalanced parentheses: unexpected ')'
'2 3'      -> ExpressionError: Missing operator before a number
'()'       -> ExpressionError: Missing operand before ')' (empty parentheses or dangling operator)
'* 3'      -> ExpressionError: Operator '*' is missing its left operand
'3 $ 4'    -> ExpressionError: Invalid character '$' at position 2
'2 + * 3'  -> ExpressionError: Operator '*' is missing its left operand
'3.5 + 1'  -> ExpressionError: Invalid character '.' at position 1
'(2)(3)'   -> ExpressionError: Missing operator before '('
'abc'      -> ExpressionError: Invalid character 'a' at position 0
'5 / 0'            -> ZeroDivisionError: Division by zero in e

# TASK 06 Banking Ledger with OOP

In [ ]:
import itertools
import math
from dataclasses import dataclass
from numbers import Real


class BankError(Exception):
    """Base class for all banking errors."""


class InvalidAmountError(BankError, ValueError):
    pass


class InsufficientFundsError(BankError):
    pass


class AccountNotFoundError(BankError):
    pass


_transaction_ids = itertools.count(1)           


@dataclass(frozen=True)
class TransactionRecord:
    """Immutable ledger entry (frozen dataclass: attributes cannot be reassigned)."""
    transaction_id: int
    action: str
    amount: float
    balance_after: float


def _validate_amount(amount, allow_zero=False):
    if isinstance(amount, bool) or not isinstance(amount, Real) or not math.isfinite(amount):
        raise InvalidAmountError(f"Amount must be a finite number, got {amount!r}")
    if amount < 0 or (amount == 0 and not allow_zero):
        raise InvalidAmountError(f"Amount must be greater than zero, got {amount!r}")
    return amount


class Account:
    """A single bank account. The balance is exposed read-only through a property."""

    def __init__(self, account_id, owner, opening_balance=0):
        if not isinstance(account_id, str) or not account_id.strip():
            raise ValueError("account_id must be a non-empty string")
        if not isinstance(owner, str) or not owner.strip():
            raise ValueError("owner must be a non-empty string")
        _validate_amount(opening_balance, allow_zero=True)
        self._account_id = account_id.strip()
        self._owner = owner.strip()
        self._balance = 0
        self._history = []
        if opening_balance > 0:
            self._apply("opening_balance", opening_balance)

  
    @property
    def account_id(self):
        return self._account_id

    @property
    def owner(self):
        return self._owner

    @property
    def balance(self):
        return self._balance

    @property
    def transactions(self):
        return tuple(self._history)          

   
    def deposit(self, amount):
        _validate_amount(amount)
        return self._apply("deposit", amount)

    def withdraw(self, amount):
        _validate_amount(amount)
        if amount > self._balance:
            raise InsufficientFundsError(
                f"Account {self._account_id}: cannot withdraw {amount}, balance is {self._balance}")
        return self._apply("withdrawal", -amount)

    def statement(self):
        lines = [f"Statement for {self._account_id} ({self._owner})",
                 f"{'ID':>4}  {'Action':<16}{'Amount':>12}{'Balance':>12}"]
        for record in self._history:
            lines.append(f"{record.transaction_id:>4}  {record.action:<16}{record.amount:>12.2f}{record.balance_after:>12.2f}")
        lines.append(f"Current balance: {self._balance:.2f}")
        return "\n".join(lines)

    def __repr__(self):
        return f"Account({self._account_id!r}, owner={self._owner!r}, balance={self._balance})"

    
    def _apply(self, action, signed_amount, transaction_id=None):
        new_balance = round(self._balance + signed_amount, 2)
        record = TransactionRecord(
            transaction_id if transaction_id is not None else next(_transaction_ids),
            action, abs(signed_amount), new_balance)
        self._balance = new_balance
        self._history.append(record)
        return record

    def _snapshot(self):
        return self._balance, len(self._history)

    def _restore(self, snapshot):
        self._balance, history_length = snapshot
        del self._history[history_length:]


class Bank:
    """Holds accounts and performs operations between them."""

    def __init__(self):
        self._accounts = {}

    def open_account(self, account_id, owner, opening_balance=0):
        if account_id in self._accounts:
            raise ValueError(f"Account {account_id!r} already exists")
        account = Account(account_id, owner, opening_balance)
        self._accounts[account.account_id] = account
        return account

    def get_account(self, account_id):
        try:
            return self._accounts[account_id]
        except KeyError:
            raise AccountNotFoundError(f"Account {account_id!r} does not exist") from None

    def deposit(self, account_id, amount):
        return self.get_account(account_id).deposit(amount)

    def withdraw(self, account_id, amount):
        return self.get_account(account_id).withdraw(amount)

    def statement(self, account_id):
        return self.get_account(account_id).statement()

    def transfer(self, from_id, to_id, amount):
        """Atomic transfer: every check happens first; if applying fails midway both accounts are restored."""
        source = self.get_account(from_id)
        target = self.get_account(to_id)
        if source is target:
            raise ValueError("Cannot transfer to the same account")
        _validate_amount(amount)
        if amount > source.balance:
            raise InsufficientFundsError(
                f"Account {from_id}: cannot transfer {amount}, balance is {source.balance}")

        transfer_id = next(_transaction_ids)   
        snapshots = (source._snapshot(), target._snapshot())
        try:
            source._apply("transfer_out", -amount, transfer_id)
            target._apply("transfer_in", amount, transfer_id)
        except Exception:
            source._restore(snapshots[0])
            target._restore(snapshots[1])
            raise
        return transfer_id



bank = Bank()
bank.open_account("A", "Ayesha", 1000)
bank.open_account("B", "Bilal", 200)

bank.transfer("A", "B", 350)
print(bank.get_account("A").balance, bank.get_account("B").balance)
assert bank.get_account("A").balance == 650 and bank.get_account("B").balance == 550

bank.deposit("A", 100)
bank.withdraw("B", 50)
print()
print(bank.statement("A"))
print()
print(bank.statement("B"))
assert bank.get_account("A").balance == 750 and bank.get_account("B").balance == 500


def expect(error_type, func, *args):
    try:
        func(*args)
    except error_type as error:
        print(f"{error_type.__name__}: {error}")
    else:
        raise AssertionError(f"expected {error_type.__name__}")


print()
expect(InvalidAmountError, bank.deposit, "A", 0)
expect(InvalidAmountError, bank.withdraw, "A", -5)
expect(InvalidAmountError, bank.transfer, "A", "B", -1)
expect(InvalidAmountError, bank.deposit, "A", "100")
expect(InsufficientFundsError, bank.withdraw, "B", 10_000)
expect(InsufficientFundsError, bank.transfer, "B", "A", 10_000)
expect(AccountNotFoundError, bank.transfer, "A", "NOPE", 10)
expect(AccountNotFoundError, bank.deposit, "NOPE", 10)
expect(ValueError, bank.transfer, "A", "A", 10)
assert (bank.get_account("A").balance, bank.get_account("B").balance) == (750, 500)  


record = bank.get_account("A").transactions[0]
try:
    record.amount = 1
except Exception as error:
    print("Immutable record:", type(error).__name__)
else:
    raise AssertionError("record should be immutable")


try:
    bank.get_account("A").balance = 10**9
except AttributeError:
    print("balance is read-only: AttributeError")


target = bank.get_account("B")
history_before = len(bank.get_account("A").transactions), len(target.transactions)


def broken_apply(*args, **kwargs):
    raise RuntimeError("simulated failure while crediting")


target._apply = broken_apply
expect(RuntimeError, bank.transfer, "A", "B", 100)
del target._apply                               
assert (bank.get_account("A").balance, bank.get_account("B").balance) == (750, 500)
assert (len(bank.get_account("A").transactions), len(target.transactions)) == history_before

print("\nTask 06: all checks passed")


650 550

Statement for A (Ayesha)
  ID  Action                Amount     Balance
   1  opening_balance      1000.00     1000.00
   3  transfer_out          350.00      650.00
   4  deposit               100.00      750.00
Current balance: 750.00

Statement for B (Bilal)
  ID  Action                Amount     Balance
   2  opening_balance       200.00      200.00
   3  transfer_in           350.00      550.00
   5  withdrawal             50.00      500.00
Current balance: 500.00

InvalidAmountError: Amount must be greater than zero, got 0
InvalidAmountError: Amount must be greater than zero, got -5
InvalidAmountError: Amount must be greater than zero, got -1
InvalidAmountError: Amount must be a finite number, got '100'
InsufficientFundsError: Account B: cannot withdraw 10000, balance is 500
InsufficientFundsError: Account B: cannot transfer 10000, balance is 500
AccountNotFoundError: Account 'NOPE' does not exist
AccountNotFoundError: Account 'NOPE' does not exist
ValueError: Cannot tra

# TASK 07 Employee Payroll Engine

In [ ]:
from abc import ABC, abstractmethod
from numbers import Real


def _non_negative(value, label):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError(f"{label} must be a number, got {value!r}")
    if value < 0:
        raise ValueError(f"{label} cannot be negative, got {value!r}")
    return value


class Employee(ABC):
    """Abstract base class: every employee type must implement calculate_pay()."""

    def __init__(self, employee_id, name):
        if not isinstance(employee_id, (int, str)) or isinstance(employee_id, bool) or str(employee_id).strip() == "":
            raise ValueError("employee_id must be a non-empty string or an integer")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name must be a non-empty string")
        self.employee_id = employee_id
        self.name = name.strip()

    @abstractmethod
    def calculate_pay(self):
        """Return the pay for one pay period."""

    def __repr__(self):
        return f"{type(self).__name__}(id={self.employee_id!r}, name={self.name!r})"


class SalariedEmployee(Employee):
    def __init__(self, employee_id, name, monthly_salary):
        super().__init__(employee_id, name)
        self.monthly_salary = _non_negative(monthly_salary, "monthly_salary")

    def calculate_pay(self):
        return round(self.monthly_salary, 2)


class HourlyEmployee(Employee):
    OVERTIME_THRESHOLD = 40
    OVERTIME_MULTIPLIER = 1.5

    def __init__(self, employee_id, name, hourly_rate, hours_worked):
        super().__init__(employee_id, name)
        self.hourly_rate = _non_negative(hourly_rate, "hourly_rate")
        self.hours_worked = _non_negative(hours_worked, "hours_worked")

    def calculate_pay(self):
        regular_hours = min(self.hours_worked, self.OVERTIME_THRESHOLD)
        overtime_hours = max(self.hours_worked - self.OVERTIME_THRESHOLD, 0)
        pay = regular_hours * self.hourly_rate + overtime_hours * self.hourly_rate * self.OVERTIME_MULTIPLIER
        return round(pay, 2)


class CommissionEmployee(Employee):
    def __init__(self, employee_id, name, base_salary, sales, commission_rate):
        super().__init__(employee_id, name)
        self.base_salary = _non_negative(base_salary, "base_salary")
        self.sales = _non_negative(sales, "sales")
        self.commission_rate = _non_negative(commission_rate, "commission_rate")
        if self.commission_rate > 1:
            raise ValueError("commission_rate must be between 0 and 1 (e.g. 0.10 for 10%)")

    def calculate_pay(self):
        return round(self.base_salary + self.sales * self.commission_rate, 2)


def generate_payroll_summary(employees):
    """Polymorphism in action: the same call works for every employee type."""
    employees = list(employees)
    for employee in employees:
        if not isinstance(employee, Employee):
            raise TypeError(f"{employee!r} is not an Employee")
    rows = [{"employee_id": e.employee_id, "name": e.name, "type": type(e).__name__, "pay": e.calculate_pay()}
            for e in employees]
    return {"rows": rows, "total_payroll": round(sum(row["pay"] for row in rows), 2)}


def format_payroll_summary(summary):
    lines = [f"{'ID':<6}{'Name':<12}{'Type':<20}{'Pay':>12}", "-" * 50]
    for row in summary["rows"]:
        lines.append(f"{str(row['employee_id']):<6}{row['name']:<12}{row['type']:<20}{row['pay']:>12.2f}")
    lines += ["-" * 50, f"{'TOTAL':<38}{summary['total_payroll']:>12.2f}"]
    return "\n".join(lines)



staff = [
    SalariedEmployee(1, "Ayesha", 3000),
    HourlyEmployee(2, "Bilal", hourly_rate=20, hours_worked=45),
    HourlyEmployee(3, "Chaman", hourly_rate=20, hours_worked=30),
    CommissionEmployee(4, "Danish", base_salary=500, sales=4000, commission_rate=0.10),
]
summary = generate_payroll_summary(staff)
print(format_payroll_summary(summary))

assert staff[1].calculate_pay() == 950         
assert staff[3].calculate_pay() == 900         
assert HourlyEmployee(9, "Exact", 20, 40).calculate_pay() == 800     
assert HourlyEmployee(9, "Zero", 20, 0).calculate_pay() == 0
assert summary["total_payroll"] == 3000 + 950 + 600 + 900

try:
    Employee(99, "Nobody")
except TypeError as error:
    print("\nDirect instantiation blocked ->", error)
else:
    raise AssertionError("Employee must be abstract")

for factory, args in [(HourlyEmployee, (5, "Neg", -20, 10)), (CommissionEmployee, (6, "Rate", 100, 100, 1.5)),
                      (SalariedEmployee, (7, "", 1000))]:
    try:
        factory(*args)
        raise AssertionError("expected ValueError")
    except ValueError as error:
        print(f"{factory.__name__}: {error}")

print("\nTask 07: all checks passed")


ID    Name        Type                         Pay
--------------------------------------------------
1     Ayesha      SalariedEmployee         3000.00
2     Bilal       HourlyEmployee            950.00
3     Chaman      HourlyEmployee            600.00
4     Danish      CommissionEmployee        900.00
--------------------------------------------------
TOTAL                                      5450.00

Direct instantiation blocked -> Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'
HourlyEmployee: hourly_rate cannot be negative, got -20
CommissionEmployee: commission_rate must be between 0 and 1 (e.g. 0.10 for 10%)
SalariedEmployee: name must be a non-empty string

Task 07: all checks passed


# TASK 08 Cache Decorator with Expiration

In [ ]:
import functools
import inspect
import threading
import time


def ttl_cache(seconds):
    """Cache a function's results for `seconds`, then recompute (time.monotonic() based).

    * Positional and keyword calls that mean the same thing share one entry: f(2, 3), f(x=2, y=3)
      and f(y=3, x=2) are all the same call (arguments are normalised with inspect.signature).
    * functools.wraps keeps __name__, __doc__, etc.
    * Unhashable arguments (lists, dicts, ...) cannot be used as a cache key. Such calls are NOT cached:
      the function is simply executed, counted as a miss and also as "uncacheable".
    * Exceptions are never cached. Statistics: wrapper.cache_info(); reset with wrapper.cache_clear().
    """
    if isinstance(seconds, bool) or not isinstance(seconds, (int, float)) or seconds <= 0:
        raise ValueError("seconds must be a positive number")

    def decorator(func):
        signature = inspect.signature(func)
        store = {}                                   
        stats = {"hits": 0, "misses": 0, "uncacheable": 0}
        lock = threading.Lock()

        def build_key(bound):
            parts = []
            for name, value in bound.arguments.items():
                if signature.parameters[name].kind is inspect.Parameter.VAR_KEYWORD:
                    value = tuple(sorted(value.items()))     
                parts.append((name, value))
            return tuple(parts)

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            try:
                bound = signature.bind(*args, **kwargs)
            except TypeError:
                return func(*args, **kwargs)                 
            bound.apply_defaults()                           
            key = build_key(bound)

            try:
                hash(key)
            except TypeError:                            
                with lock:
                    stats["misses"] += 1
                    stats["uncacheable"] += 1
                return func(*args, **kwargs)

            now = time.monotonic()
            with lock:
                entry = store.get(key)
                if entry is not None and now < entry[0]:
                    stats["hits"] += 1
                    return entry[1]
                stats["misses"] += 1

            value = func(*args, **kwargs)                     
            with lock:
                now = time.monotonic()
                for stale_key in [k for k, (expires, _) in store.items() if expires <= now]:
                    del store[stale_key]                     
                store[key] = (now + seconds, value)
            return value

        def cache_info():
            with lock:
                return {**stats, "size": len(store)}

        def cache_clear():
            with lock:
                store.clear()
                for name in stats:
                    stats[name] = 0

        wrapper.cache_info = cache_info
        wrapper.cache_clear = cache_clear
        return wrapper

    return decorator



calls = []


@ttl_cache(0.4)
def add(x, y):
    """Add two numbers."""
    calls.append((x, y))
    return x + y


assert add(x=2, y=3) == 5         
assert add(y=3, x=2) == 5         
assert add(2, 3) == 5              
assert add(2, y=3) == 5            
print(add.cache_info())
assert add.cache_info()["hits"] == 3 and add.cache_info()["misses"] == 1 and len(calls) == 1

assert add(3, 2) == 5             
assert len(calls) == 2

time.sleep(0.5)                   
assert add(2, 3) == 5              
assert len(calls) == 3
print(add.cache_info())


assert add.__name__ == "add" and add.__doc__ == "Add two numbers."
print("name:", add.__name__, "| doc:", add.__doc__)


@ttl_cache(5)
def power(base, exponent=2, **options):
    return base ** exponent


power(3)
power(3, 2)
power(base=3, exponent=2)
power(3, exponent=2, a=1, b=2)
power(3, exponent=2, b=2, a=1)
assert power.cache_info()["hits"] == 3 and power.cache_info()["misses"] == 2, power.cache_info()


@ttl_cache(5)
def total(items):
    return sum(items)


assert total([1, 2, 3]) == 6 and total([1, 2, 3]) == 6
assert total.cache_info()["uncacheable"] == 2 and total.cache_info()["hits"] == 0


attempts = []


@ttl_cache(5)
def flaky(n):
    attempts.append(n)
    if len(attempts) == 1:
        raise RuntimeError("first call fails")
    return n


try:
    flaky(1)
except RuntimeError:
    pass
assert flaky(1) == 1 and len(attempts) == 2
try:
    add(1)
except TypeError as error:
    print("Bad call still raises normally:", error)

try:
    ttl_cache(0)
except ValueError as error:
    print("Invalid ttl rejected:", error)

add.cache_clear()
assert add.cache_info() == {"hits": 0, "misses": 0, "uncacheable": 0, "size": 0}
print("\nTask 08: all checks passed")


{'hits': 3, 'misses': 1, 'uncacheable': 0, 'size': 1}


{'hits': 3, 'misses': 3, 'uncacheable': 0, 'size': 1}
name: add | doc: Add two numbers.
Bad call still raises normally: add() missing 1 required positional argument: 'y'
Invalid ttl rejected: seconds must be a positive number

Task 08: all checks passed


# TASK 09 Streaming Log Inspector

In [ ]:
import inspect
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"
VALID_LEVELS = {"DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"}


@dataclass
class LogStats:
    """Filled in while the generator runs (a generator cannot return extra values to its caller)."""
    lines_checked: int = 0      
    malformed: int = 0           


def _parse_line(line):
    """'YYYY-MM-DD HH:MM:SS | LEVEL | message' -> (timestamp, level, message), or None if malformed."""
    parts = line.rstrip("\r\n").split("|", 2)            
    if len(parts) != 3:
        return None
    timestamp, level, message = (part.strip() for part in parts)
    try:
        datetime.strptime(timestamp, TIMESTAMP_FORMAT)
    except ValueError:
        return None
    if level not in VALID_LEVELS or not message:
        return None
    return timestamp, level, message


def read_errors(path, stats=None):
    """Generator: stream the file line by line, yielding (line_number, timestamp, message) for ERROR records.

    The file is never loaded into memory. Malformed lines are skipped and counted in `stats.malformed`.
    Blank lines are ignored. (Being a generator, a missing file raises FileNotFoundError on the first next().)
    """
    if stats is None:
        stats = LogStats()
    with open(path, "r", encoding="utf-8", errors="replace") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            stats.lines_checked += 1
            parsed = _parse_line(line)
            if parsed is None:
                stats.malformed += 1
                continue
            timestamp, level, message = parsed
            if level == "ERROR":
                yield line_number, timestamp, message


def summarize_errors(path):
    """Return ({date: error_count} sorted by date, LogStats)."""
    stats = LogStats()
    counts = Counter()
    for _, timestamp, _ in read_errors(path, stats):
        counts[timestamp[:10]] += 1                      
    return dict(sorted(counts.items())), stats



data_dir = Path("task_data") / "task_09"
data_dir.mkdir(parents=True, exist_ok=True)
log_path = data_dir / "app.log"
log_path.write_text(
    "2026-11-07 10:30:00 | ERROR | Connection lost\n"
    "2026-11-07 10:31:12 | INFO | Reconnected\n"
    "this line is completely broken\n"
    "2026-11-07 11:00:00 | ERROR | Disk almost full | /dev/sda1\n"
    "2026-11-08 09:15:30 | ERROR | Timeout while calling payment API\n"
    "2026-11-08 09:20:00 | WARNING | Slow response\n"
    "2026-13-45 99:99:99 | ERROR | Impossible timestamp\n"
    "2026-11-08 12:00:00 | error | lowercase level is not valid\n"
    "\n"
    "2026-11-09 08:00:00 | ERROR | Out of memory\n",
    encoding="utf-8",
)

assert inspect.isgeneratorfunction(read_errors)          

stats = LogStats()
for record in read_errors(log_path, stats):
    print(record)
print(stats)

first = next(read_errors(log_path))
assert first == (1, "2026-11-07 10:30:00", "Connection lost")
assert [r[0] for r in read_errors(log_path)] == [1, 4, 5, 10]
assert [r[2] for r in read_errors(log_path)][1] == "Disk almost full | /dev/sda1"
assert (stats.malformed, stats.lines_checked) == (3, 9)

by_date, summary_stats = summarize_errors(log_path)
print("\nErrors per date:", by_date, "| malformed lines skipped:", summary_stats.malformed)
assert by_date == {"2026-11-07": 2, "2026-11-08": 1, "2026-11-09": 1}

empty_log = data_dir / "empty.log"
empty_log.write_text("", encoding="utf-8")
assert summarize_errors(empty_log)[0] == {}

try:
    next(read_errors(data_dir / "does_not_exist.log"))
except FileNotFoundError as error:
    print("Missing file ->", type(error).__name__)

print("\nTask 09: all checks passed")


(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:00:00', 'Disk almost full | /dev/sda1')
(5, '2026-11-08 09:15:30', 'Timeout while calling payment API')
(10, '2026-11-09 08:00:00', 'Out of memory')
LogStats(lines_checked=9, malformed=3)

Errors per date: {'2026-11-07': 2, '2026-11-08': 1, '2026-11-09': 1} | malformed lines skipped: 3
Missing file -> FileNotFoundError

Task 09: all checks passed


# TASK 10 CSV Inventory Reconciliation

In [ ]:
import csv
from pathlib import Path

INVENTORY_COLUMNS = ("sku", "name", "quantity")
MOVEMENT_COLUMNS = ("sku", "type", "units")
VALID_MOVEMENT_TYPES = ("IN", "OUT")


def _read_csv_rows(path, required_columns):
    """Return [(line_number, row_dict), ...]; header names are trimmed and lower-cased."""
    with open(path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"{path} is empty")
        reader.fieldnames = [name.strip().lower() for name in reader.fieldnames]
        missing = [column for column in required_columns if column not in reader.fieldnames]
        if missing:
            raise ValueError(f"{path} is missing column(s): {', '.join(missing)}")
        return [(reader.line_num, row) for row in reader]


def _cell(row, key):
    return (row.get(key) or "").strip()


def _load_inventory(path):
    names, stock = {}, {}
    for line_number, row in _read_csv_rows(path, INVENTORY_COLUMNS):
        sku, name, raw_quantity = _cell(row, "sku"), _cell(row, "name"), _cell(row, "quantity")
        if not sku:
            raise ValueError(f"{path} line {line_number}: SKU is empty")
        if sku in stock:
            raise ValueError(f"{path} line {line_number}: duplicate SKU '{sku}'")
        try:
            quantity = int(raw_quantity)
        except ValueError:
            raise ValueError(f"{path} line {line_number}: quantity '{raw_quantity}' is not a whole number") from None
        if quantity < 0:
            raise ValueError(f"{path} line {line_number}: quantity cannot be negative")
        names[sku], stock[sku] = name, quantity
    return names, stock                    

def reconcile_inventory(inventory_path, movements_path,
                        updated_path="updated_inventory.csv", rejected_path="rejected_movements.csv"):
    """Apply IN/OUT movements in file order; write updated_inventory.csv and rejected_movements.csv.

    A movement is rejected (and processing continues) when: the SKU is unknown, the type is not IN/OUT,
    units is not a whole number, units <= 0, or an OUT would make stock negative.
    """
    names, stock = _load_inventory(inventory_path)
    rejected, applied = [], 0

    for line_number, row in _read_csv_rows(movements_path, MOVEMENT_COLUMNS):
        sku, raw_type, raw_units = _cell(row, "sku"), _cell(row, "type"), _cell(row, "units")
        move_type = raw_type.upper()
        reason, units = None, None

        if sku not in stock:
            reason = f"unknown SKU '{sku}'"
        elif move_type not in VALID_MOVEMENT_TYPES:
            reason = f"invalid movement type '{raw_type}' (use IN or OUT)"
        else:
            try:
                units = int(raw_units)
            except ValueError:
                reason = f"units '{raw_units}' is not a whole number"
            else:
                if units <= 0:
                    reason = "units must be greater than zero"
                elif move_type == "OUT" and units > stock[sku]:
                    reason = f"OUT of {units} would make stock negative (available: {stock[sku]})"

        if reason:
            rejected.append({"line": line_number, "sku": sku, "type": raw_type, "units": raw_units, "reason": reason})
            continue
        stock[sku] += units if move_type == "IN" else -units
        applied += 1

    for output in (updated_path, rejected_path):
        Path(output).parent.mkdir(parents=True, exist_ok=True)

    with open(updated_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(INVENTORY_COLUMNS)
        for sku, quantity in stock.items():
            writer.writerow([sku, names[sku], quantity])

    with open(rejected_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["line", "sku", "type", "units", "reason"])
        writer.writeheader()
        writer.writerows(rejected)

    return {"applied": applied, "rejected": rejected, "inventory": dict(stock)}



data_dir = Path("task_data") / "task_10"
data_dir.mkdir(parents=True, exist_ok=True)
(data_dir / "inventory.csv").write_text(
    "sku,name,quantity\nP01,Widget,10\nP02,Gadget,5\nP03,Gizmo,0\n", encoding="utf-8")
(data_dir / "movements.csv").write_text(
    "sku,type,units\n"
    "P01,OUT,4\n"       
    "P01,OUT,9\n"        
    "P01,IN,2\n"        
    "P99,IN,5\n"        
    "P02,IN,0\n"       
    "P02,IN,-3\n"      
    "P02,MOVE,1\n"      
    "P02,OUT,abc\n"      
    "P02,out,5\n"        
    "P02,OUT,1\n",       
    encoding="utf-8")

result = reconcile_inventory(
    data_dir / "inventory.csv", data_dir / "movements.csv",
    data_dir / "updated_inventory.csv", data_dir / "rejected_movements.csv")

print("updated_inventory.csv\n" + (data_dir / "updated_inventory.csv").read_text(encoding="utf-8"))
print("rejected_movements.csv\n" + (data_dir / "rejected_movements.csv").read_text(encoding="utf-8"))

assert result["inventory"] == {"P01": 8, "P02": 0, "P03": 0}
assert result["applied"] == 3 and len(result["rejected"]) == 7
assert result["rejected"][0]["line"] == 3 and "negative" in result["rejected"][0]["reason"]


bad = data_dir / "bad_inventory.csv"
bad.write_text("sku,name\nP01,Widget\n", encoding="utf-8")
try:
    reconcile_inventory(bad, data_dir / "movements.csv", data_dir / "x.csv", data_dir / "y.csv")
except ValueError as error:
    print("Bad input ->", error)

print("\nTask 10: all checks passed")


updated_inventory.csv
sku,name,quantity
P01,Widget,8
P02,Gadget,0
P03,Gizmo,0

rejected_movements.csv
line,sku,type,units,reason
3,P01,OUT,9,OUT of 9 would make stock negative (available: 6)
5,P99,IN,5,unknown SKU 'P99'
6,P02,IN,0,units must be greater than zero
7,P02,IN,-3,units must be greater than zero
8,P02,MOVE,1,invalid movement type 'MOVE' (use IN or OUT)
9,P02,OUT,abc,units 'abc' is not a whole number
11,P02,OUT,1,OUT of 1 would make stock negative (available: 0)

Bad input -> task_data/task_10/bad_inventory.csv is missing column(s): quantity

Task 10: all checks passed


# TASK 11 Multi-File Student Gradebook

In [ ]:
import csv
import math
from pathlib import Path

STUDENT_COLUMNS = ("student_id", "name")
SCORE_COLUMNS = ("student_id", "subject", "marks")


def grade_for(average):
    """A >= 85, B >= 70, C >= 55, D >= 40, otherwise F."""
    for threshold, letter in ((85, "A"), (70, "B"), (55, "C"), (40, "D")):
        if average >= threshold:
            return letter
    return "F"


def _read_csv_rows(path, required_columns):
    with open(path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"{path} is empty")
        reader.fieldnames = [name.strip().lower() for name in reader.fieldnames]
        missing = [column for column in required_columns if column not in reader.fieldnames]
        if missing:
            raise ValueError(f"{path} is missing column(s): {', '.join(missing)}")
        return [(reader.line_num, row) for row in reader]


def _cell(row, key):
    return (row.get(key) or "").strip()


def _id_sort_key(student_id):
    """Numeric IDs sort numerically (2 before 10); other IDs sort as text after them."""
    if student_id.isascii() and student_id.isdigit():
        return (0, int(student_id), "")
    return (1, 0, student_id)


def build_gradebook(students_path, scores_path, report_path="report.csv"):
    """Merge students.csv + scores.csv, compute average/grade/rank and write report.csv.

    Rejected score rows (unknown student ID, marks not a number, marks outside 0-100, duplicate
    student-subject pair - the first entry is kept) are returned in result["rejected"].
    Ranking: average descending, then student ID. A student without any valid score gets grade
    "N/A" and no rank, and is listed last.
    """
    names = {}
    for line_number, row in _read_csv_rows(students_path, STUDENT_COLUMNS):
        student_id, name = _cell(row, "student_id"), _cell(row, "name")
        if not student_id or not name:
            raise ValueError(f"{students_path} line {line_number}: student_id and name are required")
        if student_id in names:
            raise ValueError(f"{students_path} line {line_number}: duplicate student_id '{student_id}'")
        names[student_id] = name

    marks_by_student = {student_id: {} for student_id in names}      
    rejected = []

    for line_number, row in _read_csv_rows(scores_path, SCORE_COLUMNS):
        student_id, subject, raw_marks = _cell(row, "student_id"), _cell(row, "subject"), _cell(row, "marks")
        reason = None
        if student_id not in names:
            reason = f"unknown student_id '{student_id}'"
        elif not subject:
            reason = "subject is empty"
        else:
            try:
                marks = float(raw_marks)
            except ValueError:
                reason = f"marks '{raw_marks}' is not a number"
            else:
                if not math.isfinite(marks) or not 0 <= marks <= 100:
                    reason = f"marks {raw_marks} must be between 0 and 100"
                elif subject.casefold() in marks_by_student[student_id]:
                    reason = f"duplicate entry for student {student_id}, subject '{subject}'"
        if reason:
            rejected.append({"line": line_number, "student_id": student_id, "subject": subject,
                             "marks": raw_marks, "reason": reason})
            continue
        marks_by_student[student_id][subject.casefold()] = marks

    scored, unscored = [], []
    for student_id, subject_marks in marks_by_student.items():
        if subject_marks:
            average = sum(subject_marks.values()) / len(subject_marks)
            scored.append({"student_id": student_id, "name": names[student_id], "subjects": len(subject_marks),
                           "average": round(average, 2), "grade": grade_for(average)})
        else:
            unscored.append({"student_id": student_id, "name": names[student_id], "subjects": 0,
                             "average": "", "grade": "N/A"})

    scored.sort(key=lambda r: (-r["average"], _id_sort_key(r["student_id"])))
    unscored.sort(key=lambda r: _id_sort_key(r["student_id"]))
    for rank, record in enumerate(scored, start=1):
        record["rank"] = rank
    for record in unscored:
        record["rank"] = ""
    report = scored + unscored

    Path(report_path).parent.mkdir(parents=True, exist_ok=True)
    with open(report_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["student_id", "name", "subjects", "average", "grade", "rank"])
        writer.writeheader()
        writer.writerows(report)

    return {"report": report, "rejected": rejected}



data_dir = Path("task_data") / "task_11"
data_dir.mkdir(parents=True, exist_ok=True)
(data_dir / "students.csv").write_text(
    "student_id,name\n101,Ali\n102,Sara\n103,Bilal\n104,Hina\n", encoding="utf-8")
(data_dir / "scores.csv").write_text(
    "student_id,subject,marks\n"
    "101,Math,90\n101,Physics,80\n101,English,85\n"
    "102,Math,95\n102,Physics,88\n"
    "103,Math,50\n103,Physics,45\n"
    "999,Math,70\n"         
    "102,Math,60\n"          
    "103,English,150\n"     
    "103,Chemistry,abc\n"   
    "101,Urdu,-1\n",        
    encoding="utf-8")

outcome = build_gradebook(data_dir / "students.csv", data_dir / "scores.csv", data_dir / "report.csv")
print("report.csv\n" + (data_dir / "report.csv").read_text(encoding="utf-8"))
print("Rejected rows:")
for item in outcome["rejected"]:
    print("  ", item)

by_id = {row["student_id"]: row for row in outcome["report"]}
assert by_id["101"]["average"] == 85.0 and by_id["101"]["grade"] == "A"
assert [row["student_id"] for row in outcome["report"]] == ["102", "101", "103", "104"]
assert [row["rank"] for row in outcome["report"]] == [1, 2, 3, ""]
assert by_id["103"]["grade"] == "D" and by_id["104"]["grade"] == "N/A"
assert len(outcome["rejected"]) == 5


assert [grade_for(x) for x in (100, 85, 84.99, 70, 69.9, 55, 54.9, 40, 39.99, 0)] == list("AABBCCDDFF")


(data_dir / "tie_students.csv").write_text("student_id,name\n10,Zoya\n2,Adam\n", encoding="utf-8")
(data_dir / "tie_scores.csv").write_text("student_id,subject,marks\n10,Math,80\n2,Math,80\n", encoding="utf-8")
tie = build_gradebook(data_dir / "tie_students.csv", data_dir / "tie_scores.csv", data_dir / "tie_report.csv")
assert [row["student_id"] for row in tie["report"]] == ["2", "10"]

print("\nTask 11: all checks passed")


report.csv
student_id,name,subjects,average,grade,rank
102,Sara,2,91.5,A,1
101,Ali,3,85.0,A,2
103,Bilal,2,47.5,D,3
104,Hina,0,,N/A,

Rejected rows:
   {'line': 9, 'student_id': '999', 'subject': 'Math', 'marks': '70', 'reason': "unknown student_id '999'"}
   {'line': 10, 'student_id': '102', 'subject': 'Math', 'marks': '60', 'reason': "duplicate entry for student 102, subject 'Math'"}
   {'line': 11, 'student_id': '103', 'subject': 'English', 'marks': '150', 'reason': 'marks 150 must be between 0 and 100'}
   {'line': 12, 'student_id': '103', 'subject': 'Chemistry', 'marks': 'abc', 'reason': "marks 'abc' is not a number"}
   {'line': 13, 'student_id': '101', 'subject': 'Urdu', 'marks': '-1', 'reason': 'marks -1 must be between 0 and 100'}

Task 11: all checks passed


# TASK 12 Custom Iterable Range Toolkit

In [ ]:
class ReversibleRange:
    """A lazy range(start, stop, step) that supports iteration, reversed(), len() and `in`.

    Semantics match the built-in range: stop is exclusive and step may be negative (never zero).
    No list is ever built - values are produced one at a time, and len()/`in` use arithmetic only.
    """

    def __init__(self, start, stop, step=1):
        for label, value in (("start", start), ("stop", stop), ("step", step)):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{label} must be an integer, got {value!r}")
        if step == 0:
            raise ValueError("step must not be zero")
        self.start, self.stop, self.step = start, stop, step

    def __len__(self):
        if self.step > 0:
            return max(0, (self.stop - self.start + self.step - 1) // self.step)
        return max(0, (self.start - self.stop - self.step - 1) // (-self.step))

    def __iter__(self):
        current = self.start
        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        remaining = len(self)
        current = self.start + (remaining - 1) * self.step      
        while remaining > 0:
            yield current
            current -= self.step
            remaining -= 1

    def __contains__(self, item):
        if isinstance(item, bool) or not isinstance(item, int):
            return False
        if self.step > 0:
            in_bounds = self.start <= item < self.stop
        else:
            in_bounds = self.stop < item <= self.start
        return in_bounds and (item - self.start) % self.step == 0

    def __repr__(self):
        return f"ReversibleRange({self.start}, {self.stop}, {self.step})"



r = ReversibleRange(2, 11, 3)
print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)
assert list(r) == [2, 5, 8]
assert list(reversed(r)) == [8, 5, 2]
assert len(r) == 3 and (5 in r) and (6 not in r) and (11 not in r)

down = ReversibleRange(10, 0, -3)
print(list(down), list(reversed(down)))
assert list(down) == [10, 7, 4, 1] and list(reversed(down)) == [1, 4, 7, 10]

assert list(ReversibleRange(5, 5)) == [] and len(ReversibleRange(5, 5)) == 0
assert list(reversed(ReversibleRange(5, 1))) == []       
assert 3 in ReversibleRange(0, 5) and 5 not in ReversibleRange(0, 5)
assert "3" not in ReversibleRange(0, 5) and 2.5 not in ReversibleRange(0, 5)

checked = 0
for start in range(-6, 7):
    for stop in range(-6, 7):
        for step in [s for s in range(-4, 5) if s != 0]:
            mine, real = ReversibleRange(start, stop, step), range(start, stop, step)
            assert list(mine) == list(real), (start, stop, step)
            assert list(reversed(mine)) == list(reversed(real)), (start, stop, step)
            assert len(mine) == len(real), (start, stop, step)
            for value in range(-9, 10):
                assert (value in mine) == (value in real), (start, stop, step, value)
            checked += 1


huge = ReversibleRange(0, 10**15, 7)
assert len(huge) == (10**15 + 6) // 7 and 7 * 123_456_789 in huge and 5 not in huge
assert next(iter(reversed(huge))) == 7 * (len(huge) - 1)

for bad_args in [(0, 5, 0), (0, 5.5), ("a", 5), (True, 5)]:
    try:
        ReversibleRange(*bad_args)
        raise AssertionError("expected an error")
    except (ValueError, TypeError) as error:
        print(f"{bad_args!r:<12} -> {type(error).__name__}: {error}")

print(f"\nTask 12: all checks passed ({checked} start/stop/step combinations match range())")


[2, 5, 8] [8, 5, 2] 3 True False
[10, 7, 4, 1] [1, 4, 7, 10]
(0, 5, 0)    -> ValueError: step must not be zero
(0, 5.5)     -> TypeError: stop must be an integer, got 5.5
('a', 5)     -> TypeError: start must be an integer, got 'a'
(True, 5)    -> TypeError: start must be an integer, got True

Task 12: all checks passed (1352 start/stop/step combinations match range())


# TASK 13 Booking Conflict Scheduler

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Tuple


@dataclass(frozen=True)
class Booking:
    booking_id: str
    room: str
    start: datetime
    end: datetime


@dataclass(frozen=True)
class BookingResult:
    accepted: bool
    booking: Optional[Booking] = None
    reason: str = ""
    conflicts: Tuple[Booking, ...] = ()


class RoomScheduler:
    """Accepts a booking only if it does not overlap an existing booking of the same room.

    Two bookings overlap when new.start < existing.end and existing.start < new.end,
    so back-to-back bookings (10:00 end / 10:00 start) are allowed.
    Use either all naive or all timezone-aware datetimes - mixing them raises TypeError.
    """

    def __init__(self):
        self._by_room = {}              
        self._booking_ids = set()

    def book(self, booking_id, room, start, end):
        if not isinstance(start, datetime) or not isinstance(end, datetime):
            raise TypeError("start and end must be datetime objects")
        if not isinstance(room, str) or not room.strip():
            raise ValueError("room must be a non-empty string")

        if end <= start:
            return BookingResult(False, reason="end must be after start")
        if booking_id in self._booking_ids:
            return BookingResult(False, reason=f"duplicate booking id '{booking_id}'")

        candidate = Booking(booking_id, room.strip(), start, end)
        existing = self._by_room.get(candidate.room, [])
        conflicts = tuple(b for b in existing if candidate.start < b.end and b.start < candidate.end)
        if conflicts:
            return BookingResult(False, reason=f"overlaps {len(conflicts)} existing booking(s)", conflicts=conflicts)

        existing.append(candidate)
        existing.sort(key=lambda b: b.start)
        self._by_room[candidate.room] = existing
        self._booking_ids.add(booking_id)
        return BookingResult(True, booking=candidate)

    def bookings_for(self, room):
        return tuple(self._by_room.get(room, ()))



def at(hour, minute=0):
    return datetime(2026, 11, 7, hour, minute)


def show(result):
    if result.accepted:
        b = result.booking
        print(f"ACCEPTED  {b.booking_id}: room {b.room} {b.start:%H:%M}-{b.end:%H:%M}")
    else:
        names = ", ".join(c.booking_id for c in result.conflicts)
        print(f"REJECTED  {result.reason}" + (f" -> conflicts with: {names}" if names else ""))


scheduler = RoomScheduler()
first = scheduler.book("B1", "A", at(9), at(10));              show(first)
second = scheduler.book("B2", "A", at(10), at(11));            show(second)       
clash = scheduler.book("B3", "A", at(9, 30), at(10, 30));      show(clash)        
assert first.accepted and second.accepted and not clash.accepted
assert [c.booking_id for c in clash.conflicts] == ["B1", "B2"]

other_room = scheduler.book("B4", "B", at(9, 30), at(10, 30)); show(other_room)  
assert other_room.accepted

inside = scheduler.book("B5", "A", at(9, 15), at(9, 45));      show(inside)       
assert not inside.accepted and [c.booking_id for c in inside.conflicts] == ["B1"]

wrapping = scheduler.book("B6", "A", at(8), at(12));           show(wrapping)    
assert not wrapping.accepted and len(wrapping.conflicts) == 2

before = scheduler.book("B7", "A", at(8), at(9));              show(before)      
after = scheduler.book("B8", "A", at(11), at(12));             show(after)       
assert before.accepted and after.accepted

bad_times = scheduler.book("B9", "A", at(14), at(14));         show(bad_times)    
reversed_times = scheduler.book("B10", "A", at(15), at(14));   show(reversed_times)
duplicate = scheduler.book("B1", "C", at(9), at(10));          show(duplicate)   
assert not bad_times.accepted and not reversed_times.accepted and not duplicate.accepted

print()
for booking in scheduler.bookings_for("A"):
    print(booking.booking_id, f"{booking.start:%H:%M}-{booking.end:%H:%M}")
assert [b.booking_id for b in scheduler.bookings_for("A")] == ["B7", "B1", "B2", "B8"]  
assert scheduler.bookings_for("Z") == ()

try:
    scheduler.book("X", "A", "09:00", "10:00")
except TypeError as error:
    print("\nNon-datetime input ->", error)

print("\nTask 13: all checks passed")


ACCEPTED  B1: room A 09:00-10:00
ACCEPTED  B2: room A 10:00-11:00
REJECTED  overlaps 2 existing booking(s) -> conflicts with: B1, B2
ACCEPTED  B4: room B 09:30-10:30
REJECTED  overlaps 1 existing booking(s) -> conflicts with: B1
REJECTED  overlaps 2 existing booking(s) -> conflicts with: B1, B2
ACCEPTED  B7: room A 08:00-09:00
ACCEPTED  B8: room A 11:00-12:00
REJECTED  end must be after start
REJECTED  end must be after start
REJECTED  duplicate booking id 'B1'

B7 08:00-09:00
B1 09:00-10:00
B2 10:00-11:00
B8 11:00-12:00

Non-datetime input -> start and end must be datetime objects

Task 13: all checks passed


# TASK 14 Command-Line Contact Manager

In [ ]:
import json
import os
import re
import tempfile
from pathlib import Path

DEFAULT_DB = "contacts.json"



class StorageError(Exception):
    """The contacts file cannot be read or written safely."""


def load_contacts(path):
    """Return the stored contacts. A missing/empty file means "no contacts yet".

    A corrupt file raises StorageError and is left untouched, so existing data is never silently lost.
    """
    path = Path(path)
    if not path.exists():
        return []
    try:
        text = path.read_text(encoding="utf-8")
    except OSError as error:
        raise StorageError(f"Cannot read '{path}': {error}") from error
    if not text.strip():
        return []
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        raise StorageError(f"'{path}' is corrupt ({error}). It was NOT modified - fix or move it first.") from error

    required = {"id", "name", "phone", "email"}
    if not isinstance(data, list) or not all(isinstance(item, dict) and required <= item.keys() for item in data):
        raise StorageError(f"'{path}' does not look like a contacts file. It was NOT modified.")
    return data


def save_contacts(path, contacts):
    """Write atomically (temp file + replace) so a crash can never leave a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, temp_name = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as temp_file:
            json.dump(contacts, temp_file, indent=2, ensure_ascii=False)
        os.replace(temp_name, path)
    except Exception as error:
        try:
            os.unlink(temp_name)
        except OSError:
            pass
        raise StorageError(f"Cannot save '{path}': {error}") from error



class ContactError(ValueError):
    """Invalid input or an impossible operation."""


class ContactNotFoundError(ContactError):
    pass


_PHONE_PATTERN = re.compile(r"^\+?\d{7,15}$")
_EMAIL_PATTERN = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


def _clean_name(name):
    if not isinstance(name, str) or not name.strip():
        raise ContactError("Name must not be empty")
    return " ".join(name.split())


def _clean_phone(phone):
    if not isinstance(phone, str) or not _PHONE_PATTERN.match(re.sub(r"[\s\-()]", "", phone)):
        raise ContactError("Phone must have 7-15 digits (optional leading +; spaces, - and () allowed)")
    return phone.strip()


def _clean_email(email):
    if not isinstance(email, str) or not _EMAIL_PATTERN.match(email.strip()):
        raise ContactError("Email address is not valid")
    return email.strip()


def _check_id(contact_id):
    if isinstance(contact_id, bool) or not isinstance(contact_id, int) or contact_id <= 0:
        raise ContactError("ID must be a positive whole number")
    return contact_id


class ContactManager:
    """CRUD operations on contacts, persisted to a JSON file after every change."""

    def __init__(self, path=DEFAULT_DB):
        self._path = Path(path)
        self._contacts = load_contacts(self._path)      

    def _commit(self, new_contacts):
        save_contacts(self._path, new_contacts)         
        self._contacts = new_contacts

    def _index_of(self, contact_id):
        _check_id(contact_id)
        for index, contact in enumerate(self._contacts):
            if contact["id"] == contact_id:
                return index
        raise ContactNotFoundError(f"No contact with ID {contact_id}")

    def add_contact(self, name, phone, email, contact_id=None):
        name, phone, email = _clean_name(name), _clean_phone(phone), _clean_email(email)
        if contact_id is None:
            contact_id = max((c["id"] for c in self._contacts), default=0) + 1
        _check_id(contact_id)
        if any(c["id"] == contact_id for c in self._contacts):
            raise ContactError(f"ID {contact_id} is already used")
        contact = {"id": contact_id, "name": name, "phone": phone, "email": email}
        self._commit([*self._contacts, contact])
        return dict(contact)

    def search(self, query):
        if not isinstance(query, str) or not query.strip():
            raise ContactError("Search text must not be empty")
        needle = query.strip().casefold()
        return [dict(c) for c in self._contacts if needle in c["name"].casefold()]

    def update_contact(self, contact_id, name=None, phone=None, email=None):
        if name is None and phone is None and email is None:
            raise ContactError("Nothing to update")
        index = self._index_of(contact_id)
        updated = dict(self._contacts[index])
        if name is not None:
            updated["name"] = _clean_name(name)
        if phone is not None:
            updated["phone"] = _clean_phone(phone)
        if email is not None:
            updated["email"] = _clean_email(email)
        new_contacts = list(self._contacts)
        new_contacts[index] = updated
        self._commit(new_contacts)
        return dict(updated)

    def delete_contact(self, contact_id):
        index = self._index_of(contact_id)
        removed = self._contacts[index]
        self._commit(self._contacts[:index] + self._contacts[index + 1:])
        return dict(removed)

    def list_contacts(self):
        return [dict(c) for c in sorted(self._contacts, key=lambda c: c["id"])]



MENU = """
===== Contact Manager =====
1) Add contact
2) Search by name
3) Update contact
4) Delete contact
5) List all contacts
0) Exit"""


def _format_contact(contact):
    return f"[{contact['id']}] {contact['name']} | {contact['phone']} | {contact['email']}"


def _read_id(raw):
    try:
        return int(raw.strip())
    except ValueError:
        raise ContactError(f"'{raw.strip()}' is not a valid numeric ID") from None


def run_menu(manager, input_fn=input, output_fn=print):
    """Menu loop. input_fn/output_fn can be replaced, which makes the CLI testable without a keyboard."""
    while True:
        output_fn(MENU)
        try:
            choice = input_fn("Choose an option: ").strip()
            if choice == "0":
                output_fn("Goodbye!")
                return
            elif choice == "1":
                raw_id = input_fn("ID (blank = automatic): ").strip()
                contact = manager.add_contact(
                    name=input_fn("Name: "), phone=input_fn("Phone: "), email=input_fn("Email: "),
                    contact_id=_read_id(raw_id) if raw_id else None)
                output_fn(f"Added: {_format_contact(contact)}")
            elif choice == "2":
                matches = manager.search(input_fn("Search name: "))
                output_fn("\n".join(_format_contact(c) for c in matches) if matches else "No contacts found.")
            elif choice == "3":
                contact_id = _read_id(input_fn("Contact ID to update: "))
                changes = {
                    "name": input_fn("New name (blank = keep): ").strip() or None,
                    "phone": input_fn("New phone (blank = keep): ").strip() or None,
                    "email": input_fn("New email (blank = keep): ").strip() or None,
                }
                output_fn(f"Updated: {_format_contact(manager.update_contact(contact_id, **changes))}")
            elif choice == "4":
                removed = manager.delete_contact(_read_id(input_fn("Contact ID to delete: ")))
                output_fn(f"Deleted: {_format_contact(removed)}")
            elif choice == "5":
                contacts = manager.list_contacts()
                output_fn("\n".join(_format_contact(c) for c in contacts) if contacts else "No contacts yet.")
            else:
                output_fn("Invalid option - please choose 0-5.")
        except ContactError as error:
            output_fn(f"Error: {error}")
        except StorageError as error:
            output_fn(f"Storage problem: {error}")
        except (EOFError, KeyboardInterrupt):
            output_fn("\nInput closed - exiting.")
            return


def main(path=DEFAULT_DB):
    try:
        manager = ContactManager(path)
    except StorageError as error:
        print(f"Cannot start: {error}")
        return
    run_menu(manager)



data_dir = Path("task_data") / "task_14"
data_dir.mkdir(parents=True, exist_ok=True)
db_path = data_dir / "contacts.json"
if db_path.exists():
    db_path.unlink()                                 

manager = ContactManager(db_path)
manager.add_contact("Ayesha", "0300-1234567", "ayesha@example.com", contact_id=1)

restarted = ContactManager(db_path)                   
print([c["name"] for c in restarted.search("aye")])
assert [c["name"] for c in restarted.search("aye")] == ["Ayesha"]        
assert [c["name"] for c in restarted.search("AYE")] == ["Ayesha"]

restarted.add_contact("Bilal Khan", "+92 321 7654321", "bilal@example.com")             
assert restarted.list_contacts()[1]["id"] == 2
restarted.update_contact(2, phone="0333-1112223")
assert restarted.search("bilal")[0]["phone"] == "0333-1112223"
restarted.delete_contact(1)
assert [c["id"] for c in ContactManager(db_path).list_contacts()] == [2]

for label, call in [
    ("duplicate ID", lambda: restarted.add_contact("Dup", "03001234567", "d@example.com", contact_id=2)),
    ("empty name", lambda: restarted.add_contact("  ", "03001234567", "d@example.com")),
    ("bad phone", lambda: restarted.add_contact("Zed", "abc", "z@example.com")),
    ("bad email", lambda: restarted.add_contact("Zed", "03001234567", "not-an-email")),
    ("unknown ID", lambda: restarted.delete_contact(99)),
    ("empty update", lambda: restarted.update_contact(2)),
    ("empty search", lambda: restarted.search("  ")),
]:
    try:
        call()
        raise AssertionError(f"{label}: expected ContactError")
    except ContactError as error:
        print(f"{label:<13} -> {error}")


corrupt_path = data_dir / "corrupt.json"
corrupt_path.write_text('[{"id": 1, "name": "Precious data", ', encoding="utf-8")
original_text = corrupt_path.read_text(encoding="utf-8")
try:
    ContactManager(corrupt_path)
    raise AssertionError("expected StorageError")
except StorageError as error:
    print("\nCorrupt file ->", error)
assert corrupt_path.read_text(encoding="utf-8") == original_text           
assert ContactManager(data_dir / "never_created.json").list_contacts() == []   


keystrokes = iter(["1", "", "Chaman", "0300 9998887", "chaman@example.com",     
                   "2", "cha",                                                    
                   "3", "3", "", "0311-5556667", "",                               
                   "4", "abc",                                                    
                   "9",                                                          
                   "5", "0"])                                                      
transcript = []
menu_manager = ContactManager(db_path)               
run_menu(menu_manager, input_fn=lambda prompt="": next(keystrokes), output_fn=transcript.append)
print("\n--- scripted menu session ---")
print("\n".join(line for line in transcript if line is not MENU))
assert any("Added: [3] Chaman" in line for line in transcript)
assert any("Updated: [3] Chaman | 0311-5556667" in line for line in transcript)
assert any(line.startswith("Error:") for line in transcript) and "Goodbye!" in transcript

print("\nTask 14: all checks passed")



['Ayesha']
duplicate ID  -> ID 2 is already used
empty name    -> Name must not be empty
bad phone     -> Phone must have 7-15 digits (optional leading +; spaces, - and () allowed)
bad email     -> Email address is not valid
unknown ID    -> No contact with ID 99
empty update  -> Nothing to update
empty search  -> Search text must not be empty

Corrupt file -> 'task_data/task_14/corrupt.json' is corrupt (Expecting property name enclosed in double quotes: line 1 column 37 (char 36)). It was NOT modified - fix or move it first.

--- scripted menu session ---
Added: [3] Chaman | 0300 9998887 | chaman@example.com
[3] Chaman | 0300 9998887 | chaman@example.com
Updated: [3] Chaman | 0311-5556667 | chaman@example.com
Error: 'abc' is not a valid numeric ID
Invalid option - please choose 0-5.
[2] Bilal Khan | 0333-1112223 | bilal@example.com
[3] Chaman | 0311-5556667 | chaman@example.com
Goodbye!

Task 14: all checks passed


# TASK 15 Integrated Order Fulfillment System

In [ ]:
import json
import os
import tempfile
import unittest
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

CENT = Decimal("0.01")


def money(value):
    return Decimal(str(value)).quantize(CENT, rounding=ROUND_HALF_UP)



class OrderError(Exception):
    """Base class for all order-system errors."""


class CatalogError(OrderError):
    pass


class UnknownProductError(OrderError):
    pass


class InvalidQuantityError(OrderError, ValueError):
    pass


class OutOfStockError(OrderError):
    pass


class OrderPersistenceError(OrderError):
    pass


def validate_quantity(quantity):
    if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity <= 0:
        raise InvalidQuantityError(f"Quantity must be a positive whole number, got {quantity!r}")
    return quantity



@dataclass(frozen=True)
class Product:
    product_id: str
    name: str
    price: Decimal



class Inventory:
    """Product catalogue + stock levels."""

    def __init__(self):
        self._products = {}
        self._stock = {}

    def add_product(self, product, stock):
        if product.product_id in self._products:
            raise CatalogError(f"Duplicate product id '{product.product_id}'")
        if isinstance(stock, bool) or not isinstance(stock, int) or stock < 0:
            raise CatalogError(f"Stock for '{product.product_id}' must be a whole number >= 0")
        self._products[product.product_id] = product
        self._stock[product.product_id] = stock

    @classmethod
    def from_json(cls, path):
        """Catalogue file: [{"id": "P1", "name": "Pen", "price": 100, "stock": 5}, ...]"""
        try:
            entries = json.loads(Path(path).read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            raise CatalogError(f"Cannot read catalogue '{path}': {error}") from error
        if not isinstance(entries, list):
            raise CatalogError("Catalogue must be a JSON list of products")
        inventory = cls()
        for entry in entries:
            try:
                price = money(entry["price"])
                if price < 0:
                    raise ValueError("price cannot be negative")
                inventory.add_product(Product(str(entry["id"]), str(entry["name"]), price), entry["stock"])
            except (KeyError, TypeError, ArithmeticError, ValueError) as error:
                raise CatalogError(f"Invalid catalogue entry {entry!r}: {error!r}") from error
        return inventory

    def get_product(self, product_id):
        try:
            return self._products[product_id]
        except KeyError:
            raise UnknownProductError(f"Unknown product '{product_id}'") from None

    def stock_of(self, product_id):
        self.get_product(product_id)
        return self._stock[product_id]

    def check_availability(self, items):
        """Validate EVERY line before anything is changed."""
        for product_id, quantity in items.items():
            validate_quantity(quantity)
            available = self.stock_of(product_id)
            if quantity > available:
                raise OutOfStockError(f"Only {available} of '{product_id}' in stock, {quantity} requested")

    def deduct(self, items):
        for product_id, quantity in items.items():
            self._stock[product_id] -= quantity

    def snapshot(self):
        return dict(self._stock)

    def restore(self, snapshot):
        self._stock = dict(snapshot)



class Cart:
    def __init__(self):
        self._items = {}
        self._discount_percent = Decimal("0")

    @property
    def items(self):
        return dict(self._items)

    @property
    def discount_percent(self):
        return self._discount_percent

    def is_empty(self):
        return not self._items

    def add(self, product_id, quantity):
        validate_quantity(quantity)
        self._items[product_id] = self._items.get(product_id, 0) + quantity

    def remove(self, product_id):
        if product_id not in self._items:
            raise UnknownProductError(f"'{product_id}' is not in the cart")
        del self._items[product_id]

    def clear(self):
        self._items.clear()
        self._discount_percent = Decimal("0")

    def set_discount(self, percent):
        if isinstance(percent, bool) or not isinstance(percent, (int, float, Decimal, str)):
            raise ValueError("Discount must be a number between 0 and 100")
        try:
            value = Decimal(str(percent))
        except ArithmeticError:
            raise ValueError("Discount must be a number between 0 and 100") from None
        if not value.is_finite() or not Decimal("0") <= value <= Decimal("100"):
            raise ValueError("Discount must be between 0 and 100 percent")
        self._discount_percent = value

    def subtotal(self, inventory):
        return sum((inventory.get_product(pid).price * qty for pid, qty in self._items.items()), Decimal("0"))

    def total(self, inventory):
        factor = (Decimal("100") - self._discount_percent) / Decimal("100")
        return money(self.subtotal(inventory) * factor)



@dataclass(frozen=True)
class Order:
    order_id: str
    lines: tuple
    subtotal: Decimal
    discount_percent: Decimal
    total: Decimal
    created_at: str

    def to_dict(self):
        return {
            "order_id": self.order_id,
            "created_at": self.created_at,
            "lines": [dict(line) for line in self.lines],
            "subtotal": float(self.subtotal),
            "discount_percent": float(self.discount_percent),
            "total": float(self.total),
        }


class OrderStore:
    """Persists orders to a JSON list on disk."""

    def __init__(self, path):
        self.path = Path(path)

    def load(self):
        if not self.path.exists():
            return []
        try:
            data = json.loads(self.path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            raise OrderPersistenceError(f"Orders file '{self.path}' is unreadable (left untouched): {error}") from error
        if not isinstance(data, list):
            raise OrderPersistenceError(f"Orders file '{self.path}' is not a JSON list (left untouched)")
        return data

    def append(self, order):
        orders = self.load()
        orders.append(order.to_dict())
        self.path.parent.mkdir(parents=True, exist_ok=True)
        handle, temp_name = tempfile.mkstemp(dir=self.path.parent, prefix=self.path.name + ".", suffix=".tmp")
        try:
            with os.fdopen(handle, "w", encoding="utf-8") as temp_file:
                json.dump(orders, temp_file, indent=2)
            os.replace(temp_name, self.path)
        except Exception as error:
            try:
                os.unlink(temp_name)
            except OSError:
                pass
            raise OrderPersistenceError(f"Cannot write orders file '{self.path}': {error}") from error


def place_order(cart, inventory, store):
    """Turn a cart into an order.

    1. validate every item (nothing is modified yet)  2. deduct stock  3. persist the order
    If step 3 fails, the stock is restored and OrderPersistenceError is raised (all-or-nothing).
    """
    if cart.is_empty():
        raise OrderError("Cannot place an order with an empty cart")

    items = cart.items
    inventory.check_availability(items)

    lines = tuple(
        {"product_id": pid, "name": inventory.get_product(pid).name,
         "unit_price": float(inventory.get_product(pid).price), "quantity": qty,
         "line_total": float(money(inventory.get_product(pid).price * qty))}
        for pid, qty in items.items())
    order = Order(
        order_id="ORD-" + uuid.uuid4().hex,                      
        lines=lines,
        subtotal=money(cart.subtotal(inventory)),
        discount_percent=cart.discount_percent,
        total=cart.total(inventory),
        created_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))

    stock_before = inventory.snapshot()
    inventory.deduct(items)
    try:
        store.append(order)
    except Exception as error:
        inventory.restore(stock_before)                           
        if isinstance(error, OrderPersistenceError):
            raise
        raise OrderPersistenceError(f"Order could not be saved, stock was rolled back: {error}") from error

    cart.clear()
    return order



class OrderSystemTests(unittest.TestCase):
    def setUp(self):
        self._tmp = tempfile.TemporaryDirectory()
        self.addCleanup(self._tmp.cleanup)
        self.dir = Path(self._tmp.name)
        (self.dir / "catalog.json").write_text(json.dumps([
            {"id": "P1", "name": "Notebook", "price": 100, "stock": 5},
            {"id": "P2", "name": "Pen", "price": 25.5, "stock": 1},
        ]), encoding="utf-8")
        self.inventory = Inventory.from_json(self.dir / "catalog.json")
        self.store = OrderStore(self.dir / "orders.json")

    def test_total_with_discount_and_stock_update(self):
        cart = Cart()
        cart.add("P1", 2)
        cart.set_discount(10)
        order = place_order(cart, self.inventory, self.store)
        self.assertEqual(order.total, Decimal("180.00"))
        self.assertEqual(self.inventory.stock_of("P1"), 3)
        saved = self.store.load()
        self.assertEqual(len(saved), 1)
        self.assertEqual(saved[0]["total"], 180.0)
        self.assertTrue(cart.is_empty())

    def test_out_of_stock_changes_nothing(self):
        cart = Cart()
        cart.add("P1", 2)                
        cart.add("P2", 3)                
        with self.assertRaises(OutOfStockError):
            place_order(cart, self.inventory, self.store)
        self.assertEqual(self.inventory.stock_of("P1"), 5)       
        self.assertEqual(self.inventory.stock_of("P2"), 1)
        self.assertEqual(self.store.load(), [])

    def test_invalid_quantities_rejected(self):
        cart = Cart()
        for bad in (0, -1, 1.5, "2", None, True):
            with self.assertRaises(InvalidQuantityError):
                cart.add("P1", bad)
        self.assertTrue(cart.is_empty())

    def test_rollback_when_persistence_fails(self):
        class BrokenStore(OrderStore):
            def append(self, order):
                raise OSError("disk full")

        cart = Cart()
        cart.add("P1", 4)
        with self.assertRaises(OrderPersistenceError):
            place_order(cart, self.inventory, BrokenStore(self.dir / "orders.json"))
        self.assertEqual(self.inventory.stock_of("P1"), 5)       
        self.assertFalse(cart.is_empty())                         

    def test_corrupt_orders_file_is_not_overwritten_and_stock_rolls_back(self):
        (self.dir / "orders.json").write_text("{ not valid json", encoding="utf-8")
        cart = Cart()
        cart.add("P1", 1)
        with self.assertRaises(OrderPersistenceError):
            place_order(cart, self.inventory, self.store)
        self.assertEqual((self.dir / "orders.json").read_text(encoding="utf-8"), "{ not valid json")
        self.assertEqual(self.inventory.stock_of("P1"), 5)

    def test_order_ids_are_unique(self):
        ids = set()
        for _ in range(3):
            cart = Cart()
            cart.add("P1", 1)
            ids.add(place_order(cart, self.inventory, self.store).order_id)
        self.assertEqual(len(ids), 3)
        self.assertEqual(len(self.store.load()), 3)
        self.assertEqual(self.inventory.stock_of("P1"), 2)

    def test_unknown_product_empty_cart_and_bad_discount(self):
        cart = Cart()
        with self.assertRaises(OrderError):
            place_order(cart, self.inventory, self.store)       
        cart.add("NOPE", 1)
        with self.assertRaises(UnknownProductError):
            place_order(cart, self.inventory, self.store)
        for bad in (-1, 101, "abc", float("nan"), True):
            with self.assertRaises(ValueError):
                cart.set_discount(bad)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(OrderSystemTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()



data_dir = Path("task_data") / "task_15"
data_dir.mkdir(parents=True, exist_ok=True)
(data_dir / "catalog.json").write_text(json.dumps([
    {"id": "P1", "name": "Notebook", "price": 100, "stock": 5},
    {"id": "P2", "name": "Pen", "price": 25.5, "stock": 20},
], indent=2), encoding="utf-8")
orders_file = data_dir / "orders.json"
if orders_file.exists():
    orders_file.unlink()

inventory = Inventory.from_json(data_dir / "catalog.json")
cart = Cart()
cart.add("P1", 2)
cart.set_discount(10)
print("\nBefore: stock P1 =", inventory.stock_of("P1"), "| cart total =", cart.total(inventory))

placed = place_order(cart, inventory, OrderStore(orders_file))
print("Order", placed.order_id, "-> total", placed.total, "| stock P1 now", inventory.stock_of("P1"))
assert placed.total == Decimal("180.00") and inventory.stock_of("P1") == 3
print("\norders.json:\n" + orders_file.read_text(encoding="utf-8"))

print("Task 15: all checks passed")


test_corrupt_orders_file_is_not_overwritten_and_stock_rolls_back (__main__.OrderSystemTests.test_corrupt_orders_file_is_not_overwritten_and_stock_rolls_back) ... 

ok


test_invalid_quantities_rejected (__main__.OrderSystemTests.test_invalid_quantities_rejected) ... 

ok


test_order_ids_are_unique (__main__.OrderSystemTests.test_order_ids_are_unique) ... 

ok


... (6 outputs left out)

test_unknown_product_empty_cart_and_bad_discount (__main__.OrderSystemTests.test_unknown_product_empty_cart_and_bad_discount) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.026s

OK



Before: stock P1 = 5 | cart total = 180.00
Order ORD-c342c4006a3e49b7b69d11d8da060a4f -> total 180.00 | stock P1 now 3

orders.json:
[
  {
    "order_id": "ORD-c342c4006a3e49b7b69d11d8da060a4f",
    "created_at": "2026-11-07T11:07:51+00:00",
    "lines": [
      {
        "product_id": "P1",
        "name": "Notebook",
        "unit_price": 100.0,
        "quantity": 2,
        "line_total": 200.0
      }
    ],
    "subtotal": 200.0,
    "discount_percent": 10.0,
    "total": 180.0
  }
]
Task 15: all checks passed
